# Index et benchmark isolés du corpus candidat

Le notebook copie LanceDB et BM25 sous `data/experiments`, applique la synchronisation incrémentale aux 4 416 chunks candidats, puis mesure `regression` et `development`. Les index réels sont contrôlés par empreinte.


In [1]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import hashlib, json, logging, shutil, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from chunker import Chunk
from embeddings import EmbeddingModel
from evaluation import load_golden_set, RetrievalEvaluator
from retriever import HybridRetriever, Reranker
from vectorstore import VectorStoreManager

logging.basicConfig(level=logging.INFO)
cfg = load_config(ROOT / 'config.yaml')


In [2]:
def tree_fingerprint(root):
    digest = hashlib.sha256()
    for path in sorted(root.rglob('*')):
        if path.is_file():
            digest.update(str(path.relative_to(root)).encode('utf-8'))
            with open(path, 'rb') as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b''):
                    digest.update(block)
    return digest.hexdigest()

experiment_root = ROOT / 'data' / 'experiments'
candidate_chunk_files = list(experiment_root.glob('chunks_candidate_*.jsonl'))
assert candidate_chunk_files, 'Exécutez d’abord le notebook 06.'
candidate_chunks_path = max(candidate_chunk_files, key=lambda path: path.stat().st_mtime)
candidate_index_dir = experiment_root / f"index_candidate_{datetime.now():%Y%m%d_%H%M%S}"
candidate_index_dir.mkdir(parents=True, exist_ok=False)

real_lancedb = cfg.vectorstore.uri_path
real_bm25 = cfg.vectorstore.bm25_index_full_path
real_lancedb_hash_before = tree_fingerprint(real_lancedb)
real_bm25_hash_before = hashlib.sha256(real_bm25.read_bytes()).hexdigest()

candidate_lancedb = candidate_index_dir / 'lancedb'
candidate_bm25 = candidate_index_dir / 'bm25_index.pkl'
shutil.copytree(real_lancedb, candidate_lancedb)
shutil.copy2(real_bm25, candidate_bm25)
print(f'Chunks candidats : {candidate_chunks_path}')
print(f'Index candidat : {candidate_index_dir}')


Chunks candidats : <PROJECT_ROOT>\data\experiments\chunks_candidate_20260803_121427.jsonl
Index candidat : <PROJECT_ROOT>\data\experiments\index_candidate_20260803_121929


In [3]:
candidate_cfg = cfg.model_copy(deep=True)
candidate_cfg.vectorstore.uri = str(candidate_lancedb)
candidate_cfg.vectorstore.bm25_index_path = str(candidate_bm25)
candidate_cfg.observability.enabled = False

with open(candidate_chunks_path, encoding='utf-8') as handle:
    candidate_chunks = [Chunk(**json.loads(line)) for line in handle if line.strip()]
print(f'{len(candidate_chunks)} chunks chargés.')

embedding_model = EmbeddingModel(candidate_cfg.embeddings)
candidate_store = VectorStoreManager(candidate_cfg)
candidate_store.load_sparse()
index_report = candidate_store.sync_incremental(candidate_chunks, embedding_model)
display(pd.Series(index_report.__dict__, name='valeur').to_frame())


4416 chunks chargés.


,valeur
changed_urls,483.000000
embedded_chunks,873.000000
reused_chunks,3543.000000
removed_chunks,916.000000
total_chunks,4416.000000
embedding_seconds,18.469312


In [4]:
reranker = Reranker(candidate_cfg.reranker.model_name, candidate_cfg.reranker.device)
candidate_retriever = HybridRetriever(
    candidate_cfg, embedding_model, candidate_store, reranker,
)
candidate_pipeline = SimpleNamespace(retriever=candidate_retriever)
all_examples = load_golden_set(candidate_cfg.evaluation.expanded_golden_set_full_path)
_ = candidate_retriever.retrieve('Quelles formations propose l’ENSEA ?')
print('Retriever candidat initialisé et échauffé.')


INFO:vectorstore:Cache RAM des vecteurs chargé : 4416 vecteurs (17.2 Mo).
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:retriever:Retrieval 4.054s | query=0.029s dense=0.031s bm25=0.003s fusion=0.000s vectors+MMR=1.086s rerank=2.904s | 5 résultat(s)


Retriever candidat initialisé et échauffé.


In [5]:
benchmark_results = {}
benchmark_details = {}
for split in ('regression', 'development'):
    examples = [example for example in all_examples if example.split == split]
    evaluator = RetrievalEvaluator(candidate_cfg.evaluation)
    summary = evaluator.evaluate(candidate_pipeline, examples)
    benchmark_results[split] = summary
    benchmark_details[split] = evaluator.last_details
    print(f'\n{split.upper()}')
    display(pd.Series(summary, name='valeur').to_frame())
    details = pd.DataFrame(evaluator.last_details)
    misses = details[details['should_answer'] & details['first_relevant_rank'].isna()]
    print(f'Échecs stricts : {len(misses)}/{len(details)}')
    display(misses[['id', 'question', 'category', 'top_title', 'top_url', 'relevant_urls']])


INFO:retriever:Retrieval 0.424s | query=0.034s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.003s rerank=0.365s | 5 résultat(s)
INFO:retriever:Retrieval 0.424s | query=0.031s dense=0.019s bm25=0.004s fusion=0.000s vectors+MMR=0.002s rerank=0.368s | 5 résultat(s)
INFO:retriever:Retrieval 0.443s | query=0.044s dense=0.018s bm25=0.007s fusion=0.000s vectors+MMR=0.002s rerank=0.372s | 5 résultat(s)
INFO:retriever:Retrieval 0.403s | query=0.031s dense=0.018s bm25=0.003s fusion=0.000s vectors+MMR=0.003s rerank=0.348s | 5 résultat(s)
INFO:retriever:Retrieval 0.416s | query=0.047s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.003s rerank=0.344s | 5 résultat(s)
INFO:retriever:Retrieval 0.456s | query=0.067s dense=0.019s bm25=0.005s fusion=0.000s vectors+MMR=0.002s rerank=0.362s | 5 résultat(s)
INFO:retriever:Retrieval 0.407s | query=0.043s dense=0.020s bm25=0.003s fusion=0.000s vectors+MMR=0.002s rerank=0.338s | 5 résultat(s)
INFO:retriever:Retrieval 0.389s | query=0.030s dense=0.


REGRESSION


,valeur
questions,20.000000
answerable_questions,20.000000
unanswerable_questions,0.000000
mrr,0.693333
mean_latency_seconds,0.416732
p50_latency_seconds,0.418770
p95_latency_seconds,0.448593
max_latency_seconds,0.455574
recall@3,0.635417
precision@3,0.316667


Échecs stricts : 2/20


,id,question,category,top_title,top_url,relevant_urls
3,reg-004,Quelles sont les conditions d'admission à l'EN...,admission,Procédures d’admissions - ENSEA,https://ensea360.ensea.ed.ci/procedures-dadmis...,[https://ensea.ed.ci/formation-initiale/commen...
8,reg-009,Quels sont les laboratoires de recherche assoc...,recherche,Partenaires Académiques - ENSEA,https://ensea.ed.ci/listing-partenaires/,[https://ensea.ed.ci/partenaires/laboratoire-d...


INFO:retriever:Retrieval 0.418s | query=0.038s dense=0.020s bm25=0.005s fusion=0.000s vectors+MMR=0.003s rerank=0.352s | 5 résultat(s)
INFO:retriever:Retrieval 0.390s | query=0.055s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.002s rerank=0.309s | 4 résultat(s)
INFO:retriever:Retrieval 0.407s | query=0.040s dense=0.018s bm25=0.001s fusion=0.000s vectors+MMR=0.002s rerank=0.346s | 5 résultat(s)
INFO:retriever:Retrieval 0.437s | query=0.034s dense=0.020s bm25=0.004s fusion=0.000s vectors+MMR=0.002s rerank=0.376s | 5 résultat(s)
INFO:retriever:Retrieval 0.451s | query=0.051s dense=0.019s bm25=0.004s fusion=0.000s vectors+MMR=0.002s rerank=0.375s | 2 résultat(s)
INFO:retriever:Retrieval 0.382s | query=0.051s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.002s rerank=0.306s | 5 résultat(s)
INFO:retriever:Retrieval 0.442s | query=0.045s dense=0.018s bm25=0.007s fusion=0.000s vectors+MMR=0.002s rerank=0.369s | 5 résultat(s)
INFO:retriever:Retrieval 0.400s | query=0.059s dense=0.


DEVELOPMENT


,valeur
questions,40.000000
answerable_questions,40.000000
unanswerable_questions,0.000000
mrr,0.770833
mean_latency_seconds,0.411131
p50_latency_seconds,0.414011
p95_latency_seconds,0.441691
max_latency_seconds,0.451484
recall@3,0.654167
precision@3,0.341667


Échecs stricts : 7/40


,id,question,category,top_title,top_url,relevant_urls
5,ext-006,Que propose le Master en Management de l'ENSEA ?,formation_initiale,À propos - ENSEA,https://ensea360.ensea.ed.ci/a-propos/,[https://ensea.ed.ci/formation-initiale/master...
7,ext-008,Que couvre la formation continue de l'ENSEA ?,formation_continue,Formation Continue / L’ENSEA s’engage à renfor...,https://ensea.ed.ci/formation-continue-lensea-...,"[https://ensea.ed.ci/formation-continue/, http..."
13,ext-014,Où consulter la liste des doctorants de l'ENSEA ?,doctorat,Admission Doctorat - ENSEA,https://ensea360.ensea.ed.ci/formation-doctora...,[https://ensea.ed.ci/formation-doctorale/docto...
14,ext-015,L'ENSEA organise-t-elle des séminaires de rech...,recherche,765106d61b6f9473,https://ensea.ed.ci/wp-content/uploads/2024/06...,[https://ensea360.ensea.ed.ci/formation-contin...
26,ext-027,Où sont indiqués les coûts des formations init...,financement,,,[https://ensea.ed.ci/formation-initiale/couts/]
27,ext-028,Où puis-je consulter les tarifs de l'ENSEA ?,financement,Client entreprise - ENSEA,https://ensea360.ensea.ed.ci/formation-continu...,[https://ensea.ed.ci/faq-items/quels-sont-les-...
29,ext-030,Quel niveau faut-il pour intégrer la filière AS ?,admission,Quel est le niveau de recrutement pour intégre...,https://ensea.ed.ci/faq-items/quel-est-le-nive...,[https://ensea.ed.ci/faq-items/quel-est-le-niv...


In [6]:
result_path = candidate_index_dir / 'benchmark_results.json'
result_path.write_text(json.dumps({
    'index_report': index_report.__dict__,
    'benchmarks': benchmark_results,
    'details': benchmark_details,
}, ensure_ascii=False, indent=2), encoding='utf-8')

assert candidate_store.dense.count() == len(candidate_chunks)
assert tree_fingerprint(real_lancedb) == real_lancedb_hash_before
assert hashlib.sha256(real_bm25.read_bytes()).hexdigest() == real_bm25_hash_before
print('Index et benchmarks candidats terminés ; LanceDB et BM25 réels sont strictement intacts.')
print(f'Résultats : {result_path}')


Index et benchmarks candidats terminés ; LanceDB et BM25 réels sont strictement intacts.
Résultats : <PROJECT_ROOT>\data\experiments\index_candidate_20260803_121929\benchmark_results.json
